In [1]:
import pandas as pd
import numpy as np
import ast

In [2]:
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

movies = movies.merge(credits,on='title')

In [3]:
movies = movies[
    ['movie_id','title','overview','genres','keywords','cast','crew']
]

In [4]:
def convert(obj):

    L = []

    for i in ast.literal_eval(obj):
        L.append(i['name'])

    return L

In [5]:
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)

In [6]:
def convert_cast(obj):

    L = []

    counter = 0

    for i in ast.literal_eval(obj):

        if counter != 3:
            L.append(i['name'])
            counter += 1
        else:
            break

    return L

In [7]:
movies['cast'] = movies['cast'].apply(convert_cast)

In [8]:
print(movies.iloc[0]['cast'])

['Sam Worthington', 'Zoe Saldana', 'Sigourney Weaver']


In [9]:
def fetch_director(obj):

    L = []

    for i in ast.literal_eval(obj):

        if i['job'] == 'Director':
            L.append(i['name'])
            break

    return L

In [10]:
movies['crew'] = movies['crew'].apply(fetch_director)

In [11]:
print(movies.iloc[0]['crew'])

['James Cameron']


In [12]:
movies['genres'] = movies['genres'].apply(lambda x:[i.replace(" ","") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x])
movies['cast'] = movies['cast'].apply(lambda x:[i.replace(" ","") for i in x])
movies['crew'] = movies['crew'].apply(lambda x:[i.replace(" ","") for i in x])

In [13]:
movies.dropna(inplace=True)

In [14]:
movies['overview'] = movies['overview'].apply(lambda x: x.split())

In [15]:
movies['genres'] = movies['genres'].apply(
    lambda x:[i.replace(" ","") for i in x]
)

movies['keywords'] = movies['keywords'].apply(
    lambda x:[i.replace(" ","") for i in x]
)

movies['cast'] = movies['cast'].apply(
    lambda x:[i.replace(" ","") for i in x]
)

movies['crew'] = movies['crew'].apply(
    lambda x:[i.replace(" ","") for i in x]
)

In [16]:
movies['tags'] = (
    movies['overview']
    + movies['genres']
    + movies['keywords']
    + movies['cast']
    + movies['crew']
)

In [17]:
movies = movies[['movie_id','title','tags']]

In [18]:
movies['tags'] = movies['tags'].apply(lambda x:" ".join(x))

In [19]:
movies['tags'] = movies['tags'].apply(lambda x:x.lower())

In [20]:
movies.head()

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...
3,49026,The Dark Knight Rises,following the death of district attorney harve...
4,49529,John Carter,"john carter is a war-weary, former military ca..."


In [21]:
print(movies.iloc[0]['tags'][:1000])

in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action adventure fantasy sciencefiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d samworthington zoesaldana sigourneyweaver jamescameron


In [22]:
print(type(movies.iloc[0]['tags']))

<class 'str'>


In [23]:
from sklearn.feature_extraction.text import CountVectorizer

In [24]:
cv = CountVectorizer(max_features=5000, stop_words='english')

In [25]:
vectors = cv.fit_transform(movies['tags']).toarray()

In [26]:
print(vectors.shape)

(4806, 5000)


In [27]:
from sklearn.metrics.pairwise import cosine_similarity

In [28]:
similarity = cosine_similarity(vectors)

In [29]:
print(similarity.shape)
print(similarity[0][:10])

(4806, 4806)
[1.         0.08740748 0.05827165 0.03823596 0.17734311 0.11357771
 0.01938917 0.16927779 0.06131393 0.0733674 ]


In [30]:
import pickle

pickle.dump(similarity, open('similarity.pkl', 'wb'))
pickle.dump(movies, open('movies.pkl', 'wb'))

In [31]:
def recommend(movie):
    movie_index = movies[movies['title'] == movie].index[0]
    distances = similarity[movie_index]

    movies_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:6]

    for i in movies_list:
        print(movies.iloc[i[0]].title)

recommend('Avatar')

Titan A.E.
Small Soldiers
Independence Day
Ender's Game
Aliens vs Predator: Requiem


In [32]:
recommend('Batman Begins')

The Dark Knight
The Dark Knight Rises
Batman
Batman
Batman & Robin


In [33]:
recommend('Toy Story')

Toy Story 2
Toy Story 3
The 40 Year Old Virgin
Heartbeeps
Max Keeble's Big Move
